### INSTALLING DEPENDANCIES
> checking the type of gemini model my API supports

## API SET UP & APPS STATE


In [97]:
import google.generativeai as genai
from google.colab import userdata
import time

# --- Gemini Engine Initialization ---
try:
    # 1. Retrieve the secret key
    # Make sure 'GOOGLE_API_KEY' is named exactly this in your 🔑 tab
    GOOGLE_API_KEY = userdata.get('GOOGLE_API_KEY')
    genai.configure(api_key=GOOGLE_API_KEY)

    # 2. Select a model from your available list
    # 'gemini-2.0-flash-lite' is the most stable free option for your account
    GEMINI_MODEL = "gemini-2.0-flash-lite"

    # 3. Initialize the model
    model = genai.GenerativeModel(model_name=f"models/{GEMINI_MODEL}")

    # 4. Anti-429 Cooldown (Wait 2 seconds to clear any active rate limits)
    print("⏳ Initializing engine...")
    time.sleep(2)

    # 5. Validation Ping
    model.generate_content("ping")
    print(f"✅ Gemini Engine Active: {GEMINI_MODEL}")

except Exception as e:
    # Fallback to the 'latest' alias if the specific version hits a 429/404
    try:
        GEMINI_MODEL = "gemini-flash-latest"
        model = genai.GenerativeModel(model_name=f"models/{GEMINI_MODEL}")
        print(f"✅ Gemini Engine Active (Backup): {GEMINI_MODEL}")
    except Exception as final_error:
        print(f"❌ Connection failed: {final_error}")
        GEMINI_MODEL = None

# --- OpenAI Engine Initialization (Optional) ---
try:
    from openai import OpenAI
    OPENAI_API_KEY = userdata.get('OPENAI_API_KEY')
    if OPENAI_API_KEY:
        oa_client = OpenAI(api_key=OPENAI_API_KEY)
        print("✅ OpenAI Engine Active: gpt-4o-mini")
    else:
        oa_client = None
except:
    oa_client = None

# --- Application State ---
TOPICS = {
    '🔄 SDLC': 'Software Development Life Cycle phases and best practices.',
    '🧩 Design Patterns': 'Creational, Structural, and Behavioral software patterns.',
    '📋 Agile & Scrum': 'Agile principles, Scrum ceremonies, and iterative delivery.',
    '📐 UML Diagrams': 'Structural and Behavioral modeling with UML.',
    '🧪 Testing': 'Testing levels, TDD, and quality assurance strategies.',
    '🌿 Version Control': 'Git workflows, branching, and CI/CD pipelines.',
}

progress = {
    'xp': 0, 'total': 0, 'correct': 0, 'chat_messages': 0,
    'topic_stats': {t: {'total': 0, 'correct': 0} for t in TOPICS},
    'history': [],
}

⏳ Initializing engine...


✅ Gemini Engine Active (Backup): gemini-flash-latest
✅ OpenAI Engine Active: gpt-4o-mini


In [98]:
from google.colab import userdata
#looking for the generated key
try:
    test_key = userdata.get('GOOGLE_API_KEY')
    if test_key:
        print(f"Key found! It starts with: {test_key[:5]}...")
    else:
        print("Key not found in Secrets.")
except Exception as e:
    print(f"Error accessing secrets: {e}")

Key found! It starts with: AIzaS...


### TUTOR SET

In [99]:
from IPython.display import display, HTML, clear_output
import ipywidgets as widgets

# --- Styling ---
display(HTML("<style>.chat-container { font-family: 'Segoe UI'; max-width: 800px; }.msg-ai { background:#fff; border:1px solid #ddd; border-radius:10px; padding:12px; margin:8px 0; font-size:13px; }.msg-user { background:#1a73e8; color:#fff; border-radius:10px; padding:12px; margin:8px 0 8px auto; font-size:13px; text-align:right; max-width:75%; }</style>"))

# --- UI Components ---
engine_toggle = widgets.Dropdown(options=['Gemini (Free)', 'OpenAI (Paid)'], description='AI Engine:')
topic_dd = widgets.Dropdown(options=list(TOPICS.keys()), description='Focus Topic:')
chat_input = widgets.Text(placeholder='Type your question here...', layout={'width': '550px'})
send_btn = widgets.Button(description='Send ↗', button_style='primary')
chat_out = widgets.Output()

conversation_history = []

def get_response(prompt, sys_msg, engine):
    if 'Gemini' in engine:
        model = genai.GenerativeModel(model_name=GEMINI_MODEL, system_instruction=sys_msg)
        return model.generate_content(prompt).text
    else:
        if not oa_client: return "OpenAI key not configured."
        resp = oa_client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[{"role": "system", "content": sys_msg}, {"role": "user", "content": prompt}]
        )
        return resp.choices[0].message.content

def handle_chat(b):
    user_q = chat_input.value.strip()
    if not user_q: return
    chat_input.value = ''

    conversation_history.append({'role': 'user', 'content': user_q})
    sys_instruction = f"You are an SE tutor. Focus: {TOPICS[topic_dd.value]}. Limit response to 5 sentences."

    with chat_out:
        clear_output(wait=True)
        display(HTML("<p style='color:gray'>AI is generating response...</p>"))

        try:
            answer = get_response(user_q, sys_instruction, engine_toggle.value)
            conversation_history.append({'role': 'assistant', 'content': answer})
            progress['xp'] += 5
            progress['chat_messages'] += 1
        except Exception as e:
            conversation_history.append({'role': 'assistant', 'content': f"Error: {str(e)}"})

        clear_output()
        for m in conversation_history:
            role_style = "msg-user" if m['role'] == 'user' else "msg-ai"
            display(HTML(f"<div class='{role_style}'>{m['content']}</div>"))

send_btn.on_click(handle_chat)
display(widgets.HBox([engine_toggle, topic_dd]), widgets.HBox([chat_input, send_btn]), chat_out)

Output()

In [100]:
from google.colab import output
output.enable_custom_widget_manager()

### RANDOM CLASS TUTORING

In [101]:
quiz_topic_dd = widgets.Dropdown(options=list(TOPICS.keys()), description='Topic:')
gen_btn = widgets.Button(description='Generate Question 🧪', button_style='success', layout={'width': '200px'})
quiz_out = widgets.Output()
ans_out = widgets.Output()
current_quiz = {}

def create_quiz(b):
    gen_btn.disabled = True
    with quiz_out: clear_output(); display(HTML("<i>Drafting technical question...</i>"))
    with ans_out: clear_output()

    prompt = f"Create a multiple choice question about {TOPICS[quiz_topic_dd.value]}. Return JSON with keys: question, options (list of 4), correct (0-3 index), explanation."

    try:
        model = genai.GenerativeModel(GEMINI_MODEL)
        # Force JSON Output
        response = model.generate_content(prompt, generation_config={"response_mime_type": "application/json"})
        data = json.loads(response.text)
        current_quiz.update(data)
        current_quiz['topic'] = quiz_topic_dd.value

        with quiz_out:
            clear_output()
            display(HTML(f"<div style='padding:15px; background:black; border-radius:10px'><h3>{data['question']}</h3></div>"))
            for i, opt in enumerate(data['options']):
                btn = widgets.Button(description=opt, layout={'width': '100%', 'margin': '4px'})
                btn.on_click(lambda b, idx=i: check_quiz(idx))
                display(btn)
    except Exception as e:
        with quiz_out: print(f"Error: {e}")
    finally:
        gen_btn.disabled = False

def check_quiz(idx):
    is_correct = (idx == current_quiz['correct'])
    progress['total'] += 1
    if is_correct:
        progress['correct'] += 1
        progress['xp'] += 20
        progress['topic_stats'][current_quiz['topic']]['correct'] += 1
    progress['topic_stats'][current_quiz['topic']]['total'] += 1

    with ans_out:
        clear_output()
        color = "green" if is_correct else "red"
        status = "🎉 Correct!" if is_correct else "❌ Not quite."
        display(HTML(f"<div style='padding:15px; border-left:5px solid {color}; background:blue'><b>{status}</b><br>{current_quiz['explanation']}</div>"))
        display(gen_btn)

gen_btn.on_click(create_quiz)
display(widgets.HBox([quiz_topic_dd, gen_btn]), quiz_out, ans_out)

Output()

Output()

### PROGRESS DASHBOARD

In [102]:
def render_dash(b=None):
    acc = int(progress['correct'] / progress['total'] * 100) if progress['total'] else 0
    html = f"""
    <div style="font-family:sans-serif; border:1px solid #e0e0e0; border-radius:12px; padding:25px; background:white; box-shadow: 0 4px 6px rgba(0,0,0,0.05)">
        <h2 style="color:#1a73e8; margin-top:0">📊 Learning Progress</h2>
        <table style="width:100%; border-collapse:collapse">
            <tr>
                <td style="padding:10px; background:#f8f9fa; border-radius:8px; text-align:center">
                    <div style="font-size:24px; font-weight:bold">{progress['xp']}</div><div style="font-size:12px">Total XP</div>
                </td>
                <td style="width:20px"></td>
                <td style="padding:10px; background:#f8f9fa; border-radius:8px; text-align:center">
                    <div style="font-size:24px; font-weight:bold">{acc}%</div><div style="font-size:12px">Accuracy</div>
                </td>
            </tr>
        </table>
        <h4 style="margin-bottom:10px; margin-top:20px">Topic Proficiency</h4>
    """
    for t, s in progress['topic_stats'].items():
        p = int(s['correct']/s['total']*100) if s['total'] else 0
        html += f"<div style='font-size:13px; margin-bottom:4px'>{t} ({s['total']} attempts)</div>"
        html += f"<div style='background:#eee; height:10px; border-radius:5px; margin-bottom:12px'><div style='background:#1a73e8; width:{p}%; height:10px; border-radius:5px'></div></div>"
    html += "</div>"

    with dash_out:
        clear_output()
        display(HTML(html))

dash_out = widgets.Output()
refresh = widgets.Button(description="🔄 Update Stats", button_style='info')
refresh.on_click(render_dash)
display(refresh, dash_out)
render_dash()

Button(button_style='info', description='🔄 Update Stats', style=ButtonStyle())

Output()

In [103]:
from google.colab import output
output.enable_custom_widget_manager()

In [104]:
from google.colab import output
output.disable_custom_widget_manager()

In [105]:
display(HTML("""
<style>
  .chat-container { font-family: 'Segoe UI', Arial, sans-serif; max-width: 760px; }
  .chat-header    { background:#185FA5; color:#fff; padding:14px 18px;
                    border-radius:10px 10px 0 0; font-size:15px; font-weight:600; }
  .chat-log       { background:#f8f9fb; border:1px solid #dde3ed;
                    border-top:none; padding:14px; min-height:260px;
                    max-height:400px; overflow-y:auto; }
  .msg-ai         { background:#fff; border:1px solid #dde3ed; border-radius:10px;
                    padding:10px 14px; margin:6px 0; font-size:13px;
                    line-height:1.65; color:#1a1a2e; max-width:88%; }
  .msg-user       { background:#185FA5; color:#fff; border-radius:10px;
                    padding:10px 14px; margin:6px 0 6px auto; font-size:13px;
                    line-height:1.65; max-width:72%; text-align:right; }
  .msg-label      { font-size:10px; color:#888; margin-bottom:2px; }
  .xp-badge       { display:inline-block; background:#e6f1fb; color:#185FA5;
                    border-radius:20px; padding:2px 10px; font-size:12px;
                    font-weight:600; margin-left:8px; }
  code            { background:#f0f0f0; padding:1px 5px; border-radius:3px;
                    font-size:12px; color:#d63384; }
</style>
"""))

# ── Widgets ───────────────────────────────────────────────────
topic_dd   = widgets.Dropdown(
    options=list(TOPICS.keys()),
    description='Topic:',
    layout=widgets.Layout(width='340px')
)
chat_input = widgets.Text(
    placeholder='Ask anything about software engineering...',
    layout=widgets.Layout(width='580px')
)
send_btn   = widgets.Button(
    description='Send ↗',
    button_style='primary',
    layout=widgets.Layout(width='100px')
)
clear_btn  = widgets.Button(
    description='Clear chat',
    button_style='',
    layout=widgets.Layout(width='100px')
)
chat_out   = widgets.Output()
status_out = widgets.Output()

conversation_history = []   # stores {role, content} dicts

def render_chat_log(history):
    """Re-renders the full conversation log as HTML."""
    html = '<div class="chat-container">'
    html += '<div class="chat-header">💬 SE Chat Tutor'
    html += f'<span class="xp-badge">⭐ {progress["xp"]} XP</span></div>'
    html += '<div class="chat-log" id="chat-log">'
    if not history:
        html += '<div class="msg-ai"><div class="msg-label">AI Tutor</div>'
        html += 'Hello! Select a topic above and ask me anything — definitions, examples, '
        html += 'interview questions, or have me walk through a concept step by step.</div>'
    for msg in history:
        if msg['role'] == 'user':
            html += f'<div style="text-align:right"><div class="msg-label" style="text-align:right">You</div>'
            html += f'<div class="msg-user">{msg["content"]}</div></div>'
        else:
            content = msg['content'].replace('\n', '<br>')
            html += f'<div class="msg-label">AI Tutor</div>'
            html += f'<div class="msg-ai">{content}</div>'
    html += '</div></div>'
    return html

def on_send(b):
    """Handles sending a message to the Claude API."""
    question = chat_input.value.strip()
    if not question:
        return
    chat_input.value = ''
    send_btn.disabled = True

    # Add user message to history
    conversation_history.append({'role': 'user', 'content': question})

    # Show thinking state
    with chat_out:
        clear_output(wait=True)
        display(HTML(render_chat_log(conversation_history) +
                     '<p style="color:#888;font-size:12px;margin-top:6px">⏳ AI is thinking...</p>'))

    # Build system prompt scoped to selected topic
    topic_name = topic_dd.value
    topic_desc = TOPICS[topic_name]
    system_prompt = f"""You are an expert software engineering tutor.
Current topic context: {topic_desc}.
Explain concepts clearly with concrete examples. Use plain text only (no markdown).
Keep responses concise: 2-4 sentences for simple questions, up to 8 for complex ones.
Be encouraging, educational, and precise."""

    try:
        response = client.messages.create(
            model=MODEL,
            max_tokens=1000,
            system=system_prompt,
            messages=conversation_history   # full history for context
        )
        reply = response.content[0].text
        conversation_history.append({'role': 'assistant', 'content': reply})

        # Update progress
        progress['xp'] += 5
        progress['chat_messages'] += 1
        progress['history'].append({
            'type': 'chat', 'topic': topic_name,
            'question': question, 'time': datetime.now().strftime('%H:%M')
        })

    except Exception as e:
        conversation_history.append({'role': 'assistant', 'content': f'Error: {str(e)}'})

    send_btn.disabled = False
    with chat_out:
        clear_output(wait=True)
        display(HTML(render_chat_log(conversation_history)))

def on_clear(b):
    """Clears the conversation history."""
    conversation_history.clear()
    with chat_out:
        clear_output(wait=True)
        display(HTML(render_chat_log([])))

def on_enter(widget):
    """Allows pressing Enter to send."""
    on_send(None)

send_btn.on_click(on_send)
clear_btn.on_click(on_clear)
chat_input.on_submit(on_enter)

# ── Layout & Display ──────────────────────────────────────────
display(widgets.HBox([topic_dd]))
display(widgets.HBox([chat_input, send_btn, clear_btn],
                     layout=widgets.Layout(margin='8px 0')))
with chat_out:
    display(HTML(render_chat_log([])))
display(chat_out)

/tmp/ipykernel_27720/347410091.py:135: DeprecationWarning: on_submit is deprecated. Instead, set the .continuous_update attribute to False and observe the value changing with: mywidget.observe(callback, 'value').
  chat_input.on_submit(on_enter)


Output()